# AVP Quick Start – Zero Tokens Between Agents

**Agent Vector Protocol** transfers KV-cache between agents instead of text. This notebook compares three modes on 10 GSM8K math problems:

- **Direct** – single agent, no pipeline
- **Latent (AVP)** – Agent A thinks (builds KV-cache), Agent B generates from it. Zero text tokens between agents.
- **Text chain** – Agent A generates text, Agent B re-processes it as prompt context

Runs on a free Colab T4 GPU in ~3 minutes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorArc/avp-python/blob/main/notebooks/avp_quick_start.ipynb)

In [ ]:
!pip install -q avp datasets

## 1. Load Model and Dataset

We load a small (1.5B parameter) instruction-tuned model that fits comfortably on a free T4 GPU, and grab 10 grade-school math problems from [GSM8K](https://huggingface.co/datasets/openai/gsm8k) to test on.

In [ ]:
import time
import torch
from avp import HuggingFaceConnector
from datasets import load_dataset

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
connector = HuggingFaceConnector.from_pretrained(MODEL)
tokenizer = connector.tokenizer

# Load 10 GSM8K problems
ds = load_dataset("openai/gsm8k", "main", split="test")
samples = list(ds.select(range(10)))

print(f"Model: {MODEL}")
print(f"Device: {connector.device}")
if str(connector.device) == "cpu":
    print("WARNING: Running on CPU \u2013 expect very slow inference.")
    print("This demo needs a GPU: Runtime > Change runtime type > T4 GPU")
print(f"Problems loaded: {len(samples)}")

## 2. Evaluation Helpers

We need a way to extract numeric answers from model output (models may use `\boxed{}`, `####`, or "the answer is ..." formats) and compare them to the gold answers. These helpers match the evaluation logic used in our [full benchmarks](https://github.com/VectorArc/avp-python/blob/main/docs/BENCHMARKS.md).

In [ ]:
import re

_NUM = r"[-+]?\d+(?:,\d{3})*(?:\.\d+)?"

def extract_answer(text):
    """Extract numeric answer from model output.

    Matches the benchmark evaluator (benchmarks/gsm8k/evaluate.py).
    Priority: \\boxed{} > #### > \"the answer is\" > last number.
    """
    # 1. \boxed{...} (take last occurrence)
    boxes = re.findall(r"\\boxed\{([^}]*)\}", text)
    if boxes:
        content = boxes[-1]
        inner = re.search(r"\\text\{([^}]*)\}", content)
        if inner:
            content = inner.group(1)
        number = re.search(_NUM, content)
        if number:
            return number.group(0).replace(",", "")

    # 2. #### <number> (GSM8K gold format)
    hashes = re.findall(r"####\s*(" + _NUM + r")", text)
    if hashes:
        return hashes[-1].replace(",", "")

    # 3. \"the answer is <number>\"
    answer_match = re.findall(
        r"(?:the\s+)?(?:final\s+)?answer\s+is\s*:?\s*\$?\s*(" + _NUM + r")",
        text, re.IGNORECASE,
    )
    if answer_match:
        return answer_match[-1].replace(",", "")

    # 4. Last number in text
    numbers = re.findall(_NUM, text)
    return numbers[-1].replace(",", "") if numbers else ""

def normalize_answer(ans):
    """Normalize for comparison: strip trailing .0, commas, whitespace."""
    if not ans:
        return ""
    ans = ans.strip().replace(",", "")
    if ans.endswith(".0"):
        ans = ans[:-2]
    return ans

def render_prompt(tokenizer, role_msg, question):
    """Build a chat prompt for the model."""
    messages = [
        {"role": "system", "content": role_msg},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

## 3. Run the Comparison

For each problem, we run three modes:

1. **Direct** – A single solver agent generates the answer. No pipeline, no collaboration.
2. **Latent (AVP)** – Agent A (Researcher) calls `connector.think()` to build a KV-cache with 20 latent reasoning steps. Agent B (Solver) calls `connector.generate()` with that KV-cache as context. **Zero text tokens** pass between agents.
3. **Text chain** – Agent A generates a full text analysis. Agent B receives it as part of its prompt. All of Agent A's output must be serialized, re-tokenized, and re-processed by Agent B.

The key difference: latent mode transfers *computation* (KV-cache), text mode transfers *text* (tokens).

In [ ]:
RESEARCHER_PROMPT = "You are a math researcher. Analyze the problem and identify the approach."
SOLVER_PROMPT = "You are a math solver. Solve the problem step by step and give the final numeric answer."
LATENT_STEPS = 20

results = {"direct": [], "latent": [], "text": []}
timings = {"direct": [], "latent": [], "text": []}
inter_agent_tokens = {"direct": [], "latent": [], "text": []}

for i, sample in enumerate(samples):
    question = sample["question"]
    gold = normalize_answer(extract_answer(sample["answer"]))
    print(f"\nProblem {i+1}: {question[:80]}...")
    print(f"  Gold answer: {gold}")

    # --- Direct (single agent, no chain) ---
    t0 = time.perf_counter()
    prompt = render_prompt(tokenizer, SOLVER_PROMPT, question)
    answer = connector.generate(prompt, max_new_tokens=512, do_sample=False)
    dt = time.perf_counter() - t0
    pred = normalize_answer(extract_answer(answer))
    correct = pred == gold
    results["direct"].append(correct)
    timings["direct"].append(dt)
    inter_agent_tokens["direct"].append(0)
    print(f"  Direct:  {pred:>8s} {'OK' if correct else 'WRONG'}  ({dt:.1f}s)")

    # --- Latent (AVP: Agent A thinks, Agent B generates) ---
    # Agent A (Researcher) thinks about the problem -- builds KV-cache via latent steps.
    # Agent B (Solver) generates the answer using Agent A's KV-cache.
    # Zero text tokens pass between agents -- only KV-cache is transferred.
    t0 = time.perf_counter()
    researcher_prompt = render_prompt(tokenizer, RESEARCHER_PROMPT, question)
    context = connector.think(researcher_prompt, steps=LATENT_STEPS)
    solver_prompt = render_prompt(tokenizer, SOLVER_PROMPT, question)
    answer = connector.generate(solver_prompt, context=context, max_new_tokens=512, do_sample=False)
    dt = time.perf_counter() - t0
    pred = normalize_answer(extract_answer(answer))
    correct = pred == gold
    results["latent"].append(correct)
    timings["latent"].append(dt)
    inter_agent_tokens["latent"].append(0)  # zero text tokens between agents
    print(f"  Latent:  {pred:>8s} {'OK' if correct else 'WRONG'}  ({dt:.1f}s, 0 inter-agent tokens)")

    # --- Text (2-agent text chain) ---
    # Agent A generates a full text analysis. Agent B receives it as prompt context.
    # All of Agent A's output must be serialized, re-tokenized, and re-processed.
    t0 = time.perf_counter()
    researcher_prompt = render_prompt(tokenizer, RESEARCHER_PROMPT, question)
    researcher_output = connector.generate(researcher_prompt, max_new_tokens=512, do_sample=False)
    researcher_tok_count = len(tokenizer.encode(researcher_output))
    solver_prompt = render_prompt(tokenizer, SOLVER_PROMPT,
        f"{question}\n\nResearcher's analysis:\n{researcher_output}")
    answer = connector.generate(solver_prompt, max_new_tokens=512, do_sample=False)
    dt = time.perf_counter() - t0
    pred = normalize_answer(extract_answer(answer))
    correct = pred == gold
    results["text"].append(correct)
    timings["text"].append(dt)
    inter_agent_tokens["text"].append(researcher_tok_count)
    print(f"  Text:    {pred:>8s} {'OK' if correct else 'WRONG'}  ({dt:.1f}s, {researcher_tok_count} inter-agent tokens)")

## 4. Results

Summary table comparing accuracy, wall-clock time, and inter-agent token transfer across all three modes.

In [ ]:
n = len(samples)
print(f"\n{'='*60}")
print(f"AVP Quick Start -- {MODEL}, {n} GSM8K problems")
print(f"{'='*60}")
print(f"\n{'Mode':<12} {'Accuracy':>10} {'Time/q':>10} {'Agent A->B':>14}")
print(f"{'-'*12} {'-'*10} {'-'*10} {'-'*14}")
for mode in ["direct", "latent", "text"]:
    acc = sum(results[mode]) / n
    avg_time = sum(timings[mode]) / n
    total_inter = sum(inter_agent_tokens[mode])
    if mode == "direct":
        transfer = "(single agent)"
    elif mode == "latent":
        transfer = "0 tokens (KV)"
    else:
        transfer = f"{total_inter:,} tokens"
    print(f"{mode:<12} {acc:>9.0%} {avg_time:>9.1f}s {transfer:>14}")

# Speedup
avg_latent = sum(timings["latent"]) / n
avg_text = sum(timings["text"]) / n
speedup = avg_text / avg_latent if avg_latent > 0 else 0
total_text_inter = sum(inter_agent_tokens["text"])

print(f"\nLatent pipeline: {speedup:.1f}x faster than text chain.")
print(f"Text chain passed {total_text_inter:,} tokens between agents across {n} problems.")
print(f"Latent mode passed zero \u2013 computation transfers via KV-cache.")
print(f"\nNote: {n} problems is a quick demo (greedy decoding, deterministic).")
print(f"See 'What\\'s Next' for full-scale results on 7B models with n=200.")
print(f"\nLearn more: https://github.com/VectorArc/avp-python")

## What's Next

This demo runs a 1.5B model on a free T4 GPU with 10 problems (greedy decoding for deterministic results). Full benchmark results on 7B models (n=164-500, NVIDIA A100) show:

- **HumanEval** (code gen): Latent 67.1% vs Text 53.0% – **+12.4pp across 4 seeds** (p=0.004)
- **GSM8K** (math): Latent 90.5% vs Text 87.0%, pipeline runs **2x faster**
- **DebugBench** (debugging): All modes neutral, pipeline runs **3x faster** (7.6s vs 22.8s)

See the [full benchmarks](https://github.com/VectorArc/avp-python/blob/main/docs/BENCHMARKS.md) for details.